# Run League: Fitness Plan AI (end to end)

The complete fitness plan AI in one notebook: it classifies a runner's level, then generates their training plan.

**Two components:**
1. **Level classifier (machine learning).** Unsupervised clustering groups runners by how they actually train, giving data-driven levels; a **Random Forest** (same family as the risk model) then predicts a new runner's level — beginner / intermediate / advanced.
2. **Plan generator (rule-based expert system).** The level drives a week-by-week schedule built from standard coaching heuristics and evidence-backed rules (10% build, 4th-week cutback, long run ≈ ⅓, taper). This is a faithful Python mirror of the production `PlanBuilder.js`.

Together: **features → ML classifier → rule-based plan.**

Input: `runs_clean.csv` from the data-prep notebook.

**Honest scope (read this):** there is no ground-truth "level" label in the data, so the levels come from clustering and the Random Forest learns to reproduce them (a standard *cluster-then-classify* pipeline). Its value is a deployable model plus evidence of *which* inputs decide level — not proof of an external truth. Cluster separation is weak (step 3), so three levels is a practical choice the data supports, not a law. The plan generator's numbers are standard coaching heuristics or evidence-backed (not individually proven RCT values); the backend JS file documents each one's provenance.

## 1. Load and build per-user features

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DIR='/content/drive/MyDrive/RunLeague/'

import pandas as pd, numpy as np
df=pd.read_csv(DIR+'runs_clean.csv', parse_dates=['date'])
df['iso_week']=df.date.dt.strftime('%G-%V')

g=df.groupby('user_id')
active_weeks=g.iso_week.nunique().clip(lower=1)   # weeks in which they actually ran
feat=pd.DataFrame({
    'weekly_km':      g.distance_km.sum()/active_weeks,   # volume when training
    'runs_per_week':  g.size()/active_weeks,              # consistency
    'longest_run_km': g.distance_km.max(),
    'avg_pace':       g.pace_min_per_km.mean(),           # speed (min/km, lower=faster)
})
feat['tenure_months']=((g.date.max()-g.date.min()).dt.days/30.44).clip(lower=0.3)
feat=feat.replace([np.inf,-np.inf],np.nan).dropna()

FEATURES=['weekly_km','runs_per_week','longest_run_km','avg_pace','tenure_months']
print(f'{len(feat)} runners')
feat[FEATURES].describe().round(1).T[['mean','25%','50%','75%','max']]

## 2. Scale the features

Volume is right-skewed (a few very high-mileage runners), so we log it first, then standardise so every feature counts equally in the distance the clustering uses.

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
X=scaler.fit_transform(np.log1p(feat[FEATURES]))

## 3. How many levels does the data actually support?

Silhouette score measures how cleanly the runners separate into *k* groups (higher = cleaner). We check k = 2..5 before committing to three.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
INK,MUTED,BAR='#1f2328','#6e7781','#3b6fb6'
plt.rcParams.update({'axes.spines.top':False,'axes.spines.right':False,'axes.edgecolor':MUTED,
    'xtick.color':MUTED,'ytick.color':MUTED,'axes.labelcolor':INK,'axes.grid':True,'grid.color':'#e6e8eb'})

ks=range(2,6); sils=[silhouette_score(X,KMeans(n_clusters=k,n_init=10,random_state=0).fit_predict(X)) for k in ks]
for k,s in zip(ks,sils): print(f'k={k}: silhouette {s:.3f}')
plt.figure(figsize=(6,3.5))
plt.plot(list(ks),sils,'o-',color=BAR,linewidth=2)
plt.axvline(3,color='#c0392b',linestyle='--',linewidth=1)
plt.title('Cluster separation by number of groups'); plt.xlabel('number of groups (k)'); plt.ylabel('silhouette (higher = cleaner)')
plt.tight_layout(); plt.show()
print('\nStructure is weak (scores are low), so the levels are not sharply separated.')
print('We use k=3 for practical plan differentiation; the profiles below support it.')

## 4. Cluster into three levels and profile them

In [ ]:
km=KMeans(n_clusters=3,n_init=10,random_state=0).fit(X)
feat['cluster']=km.labels_
# Order clusters into beginner<intermediate<advanced by training load (volume + consistency).
rank=feat.groupby('cluster')[['weekly_km','runs_per_week']].mean().rank().sum(axis=1)
order=rank.sort_values().index.tolist()
LEVELS={order[0]:'beginner',order[1]:'intermediate',order[2]:'advanced'}
feat['level']=feat.cluster.map(LEVELS)

profile=feat.groupby('level')[FEATURES].median().reindex(['beginner','intermediate','advanced']).round(1)
profile['runners']=feat.level.value_counts().reindex(['beginner','intermediate','advanced'])
print('Median runner in each learned level:')
profile

## 5. Visualise the groups

Two principal components (a 2-D shadow of the five features) just to see the clusters. Axes are abstract; the separation is the point.

In [ ]:
from sklearn.decomposition import PCA
P=PCA(n_components=2).fit_transform(X)
colors={'beginner':'#3b6fb6','intermediate':'#e08a1e','advanced':'#2a9d5c'}
plt.figure(figsize=(6.5,5))
for lv in ['beginner','intermediate','advanced']:
    m=feat.level.values==lv
    plt.scatter(P[m,0],P[m,1],s=12,alpha=.5,label=lv,color=colors[lv],linewidths=0)
plt.legend(frameon=False); plt.title('Runners in two principal components'); plt.xlabel('PC1'); plt.ylabel('PC2')
plt.tight_layout(); plt.show()

## 6. The deployable model: Random Forest level classifier

Now the supervised part. The cluster labels are the targets; a Random Forest learns to predict a runner's level from their five features. We hold out 25% of runners to measure it honestly.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

y=feat.level.values; Xf=feat[FEATURES].values
Xtr,Xte,ytr,yte=train_test_split(Xf,y,test_size=0.25,random_state=0,stratify=y)
rf=RandomForestClassifier(n_estimators=200,max_depth=6,random_state=0).fit(Xtr,ytr)
pred=rf.predict(Xte)
print(f'Accuracy on held-out runners: {accuracy_score(yte,pred):.1%}')

labels=['beginner','intermediate','advanced']
fig,ax=plt.subplots(figsize=(4.5,4))
ConfusionMatrixDisplay(confusion_matrix(yte,pred,labels=labels),display_labels=labels).plot(ax=ax,cmap='Blues',colorbar=False)
ax.set_title('Predicted vs actual level'); plt.tight_layout(); plt.show()
print('Mistakes are only between neighbouring levels — never beginner vs advanced.')

## 7. Which features decide the level?

In [ ]:
imp=sorted(zip(FEATURES,rf.feature_importances_),key=lambda t:-t[1])
plt.figure(figsize=(6,3))
plt.barh([f for f,_ in imp][::-1],[i for _,i in imp][::-1],color=BAR)
plt.title('What the model uses to decide level'); plt.xlabel('importance')
plt.tight_layout(); plt.show()
for f,i in imp: print(f'  {f:16s} {i:.2f}')
print('\nWeekly volume and consistency dominate; tenure barely matters here —')
print('because these Endomondo runners are mostly experienced, so tenure has little spread.')

## 8. Save the model and the level profile

In [ ]:
import joblib, json
joblib.dump({'model':rf,'features':FEATURES}, DIR+'level_classifier_rf.joblib')
# Data-derived thresholds (the 25th/75th percentile of volume per learned level) for the
# lightweight rule version in the Node backend, so the two agree.
bounds={lv:{'weekly_km_median':float(feat[feat.level==lv].weekly_km.median()),
            'runs_per_week_median':float(feat[feat.level==lv].runs_per_week.median())}
        for lv in labels}
json.dump({'levels':bounds,'feature_importance':dict(imp),
           'heldout_accuracy':float(accuracy_score(yte,pred))},
          open(DIR+'level_model_summary.json','w'), indent=2)
print(json.dumps(bounds, indent=2))
print('\nSaved level_classifier_rf.joblib and level_model_summary.json')

### Export the forest for the backend

`joblib` above saves the model for Python. The Run League backend is Node, and a trained
forest is only a set of thresholds and child indices — nothing in *predicting* needs Python.
So the same model is also written as JSON and evaluated directly in the backend
(`backend/control/RandomForest.js`), which removes a whole Python service from the
deployment. The backend's tests check the two agree exactly, class probabilities included.

Copy the file this writes to `backend/ml/level_classifier_rf.json`.

In [ ]:
import json

FEATURE_SPEC_VERSION = 1   # bump ONLY if a feature DEFINITION below changes

def export_model_json(forest, features, path, training_data=None):
    """Write a fitted RandomForestClassifier as JSON the Node backend can evaluate."""
    trees = []
    for est in forest.estimators_:
        t = est.tree_
        # value is (n_nodes, n_outputs, n_classes); single-output classifier, so drop that axis
        value = t.value.reshape(t.value.shape[0], -1)
        trees.append({'left': t.children_left.tolist(),
                      'right': t.children_right.tolist(),
                      'feature': t.feature.tolist(),
                      'threshold': t.threshold.tolist(),
                      'value': [r.tolist() for r in value]})
    stats = None
    if training_data is not None:
        stats = {n: {'min': float(training_data[n].min()),
                     'max': float(training_data[n].max()),
                     'median': float(training_data[n].median())} for n in features}
    model = {'format': 'runleague-rf-1',
             'featureSpecVersion': FEATURE_SPEC_VERSION,
             'features': list(features),
             'classes': [str(c) for c in forest.classes_],
             'trees': trees,
             'featureStats': stats,
             'meta': {'n_estimators': len(forest.estimators_)}}
    with open(path, 'w') as f:
        json.dump(model, f)
    return path

out = export_model_json(rf, FEATURES, DIR + 'level_classifier_rf.json',
                        training_data=feat[FEATURES])
import os
print(f'Wrote {out}  ({os.path.getsize(out)/1024:.0f} KB, {len(rf.estimators_)} trees)')
print('Classes in order:', list(rf.classes_))
print('Features in order:', FEATURES)
print('Feature spec version:', FEATURE_SPEC_VERSION,
      '- the backend refuses a model whose version it does not implement')
print()
print('Copy this file to backend/ml/level_classifier_rf.json in the project.')

# A few predictions to paste into the backend if you ever want to spot-check the port.
import numpy as np
sample = feat[FEATURES].head(3)
for (idx, row), pred in zip(sample.iterrows(), rf.predict(sample)):
    print(dict(row.round(3)), '->', pred)


## 9. End to end: classify a runner, then build their plan

The two pieces join here: the **Random Forest** (above) decides the level, and the plan builder turns that level into a week-by-week schedule.

The function below is a **faithful Python mirror of the production code** (`backend/control/PlanBuilder.js`) — same constants, same logic, verified to produce identical output. The JavaScript in the backend is the source of truth; this copy is for demonstrating the full pipeline in one place. Every number in it is a named standard coaching heuristic or evidence-backed (the 10% build, the 4th-week cutback, the long run ≈ ⅓, the taper); see the JS file's header for the provenance of each.

In [ ]:
import math
WEEKLY_BUILD=1.10; CUTBACK_EVERY=4; CUTBACK_FACTOR=0.70
LONG_RUN_SHARE=0.33; LONG_RUN_SHARE_2DAY=0.50; TAPER=[0.60,0.50]
SPIKE_LIMIT=1.5; NO_HISTORY_START=12
LEVEL_PROFILES={
 'beginner':{'maxFrequency':3,'qualityPerWeek':0,'startQualityWeek':math.inf,'runWalkWeeks':3},
 'intermediate':{'maxFrequency':5,'qualityPerWeek':1,'startQualityWeek':3,'runWalkWeeks':0},
 'advanced':{'maxFrequency':6,'qualityPerWeek':2,'startQualityWeek':2,'runWalkWeeks':0},
}
DAY_PATTERNS={2:[2,6],3:[2,4,6],4:[2,3,5,7],5:[1,2,4,6,7],6:[1,2,3,5,6,7],7:[1,2,3,4,5,6,7]}
clamp=lambda n,lo,hi:max(lo,min(hi,n)); r1=lambda n:round(n,1); r2=lambda n:round(n,2)

def _opening(recent): return clamp(recent,5,120) if recent>0 else NO_HISTORY_START
def _weekly_volumes(total,opening,race,first_cut):
    build_weeks=max(1,total-len(TAPER)) if race else total
    vols=[]; step=0; peak=opening
    for wk in range(1,total+1):
        if wk>build_weeks:
            into=wk-build_weeks; vols.append(peak*(TAPER[into-1] if into-1<len(TAPER) else TAPER[-1])); continue
        if wk==1 and first_cut: vols.append(opening*CUTBACK_FACTOR); continue
        lvl=opening*WEEKLY_BUILD**min(step,11)
        if wk%CUTBACK_EVERY==0: vols.append(lvl*CUTBACK_FACTOR)
        else: vols.append(lvl); peak=max(peak,lvl); step+=1
    return vols
def _quality_days(days,count):
    if count<=0 or len(days)<3: return set()
    cand=days[:-1]; picked=set()
    if count>=1: picked.add(cand[len(cand)//2])
    if count>=2: picked.add(cand[0])
    return picked
def _sessions(vol,freq,wk,recovery,taper,goal,prof,paces):
    days=DAY_PATTERNS.get(clamp(freq,2,7),DAY_PATTERNS[3])
    long_share=LONG_RUN_SHARE_2DAY if freq<=2 else LONG_RUN_SHARE
    long_km=r1(vol*long_share); easy=r1((vol-long_km)/(len(days)-1)) if len(days)>1 else 0
    qon=wk>=prof['startQualityWeek'] and not recovery and not taper
    qdays=_quality_days(days,prof['qualityPerWeek']) if qon else set()
    long_day=days[-1]; run_walk=wk<=prof['runWalkWeeks']; out=[]; qused=0
    def wp(s,t):
        if paces and paces.get(t) is not None: s['targetPaceMinPerKm']=r2(paces[t])
        return s
    for d in days:
        if d==long_day: out.append(wp({'dayOfWeek':d,'sessionType':'long','distanceKm':long_km},'long'))
        elif d in qdays:
            iv=(goal=='race_prep' or prof['qualityPerWeek']>=2) and qused%2==0; qused+=1
            out.append(wp({'dayOfWeek':d,'sessionType':'intervals' if iv else 'tempo','distanceKm':easy},'intervals' if iv else 'tempo'))
        else: out.append(wp({'dayOfWeek':d,'sessionType':'recovery' if recovery else 'easy','distanceKm':easy},'easy'))
    return sorted(out,key=lambda s:s['dayOfWeek'])
def build_plan(goalType='general_fitness',targetDistanceKm=None,weeklyFrequency=3,durationWeeks=8,
               recentWeeklyKm=0,riskLevel=None,level='intermediate',paces=None):
    prof=LEVEL_PROFILES.get(level,LEVEL_PROFILES['intermediate'])
    total=clamp(round(durationWeeks) or 8,1,52)
    freq=clamp(round(weeklyFrequency) or 3,2,prof['maxFrequency'])
    if riskLevel=='high': freq=clamp(freq-1,2,prof['maxFrequency'])
    race=goalType=='race_prep' and total>=4; opening=_opening(recentWeeklyKm)
    vols=_weekly_volumes(total,opening,race,riskLevel=='high')
    weeks=[]
    for wk in range(1,total+1):
        taper=race and wk>total-len(TAPER); recovery=(not taper) and wk%CUTBACK_EVERY==0
        weeks.append({'weekNumber':wk,'targetDistanceKm':r1(vols[wk-1]),
            'sessions':_sessions(r1(vols[wk-1]),freq,wk,recovery,taper,goalType,prof,paces)})
    return {'level':level,'weeks':weeks}


### Pick a runner, classify them, generate the plan

We take a real runner from the dataset, let the Random Forest predict their level from their features, then build a 12-week 10 km plan. (Paces would come from the fitness-estimate notebook; omitted here, so sessions show distances only.)

In [ ]:
# one real runner (change the index to try others)
who = feat.index[7]
row = feat.loc[who]
X_one = row[FEATURES].values.reshape(1, -1)
level = rf.predict(X_one)[0]
print(f'Runner {who}: weekly_km={row.weekly_km:.0f}, runs/week={row.runs_per_week:.1f}, '
      f'longest={row.longest_run_km:.0f} km, tenure={row.tenure_months:.0f} mo')
print(f'Random Forest level: {level.upper()}')

plan = build_plan(goalType='race_prep', targetDistanceKm=10, durationWeeks=12,
                  weeklyFrequency=4, recentWeeklyKm=float(row.weekly_km), level=level)

# weekly volume curve
vols = [w['targetDistanceKm'] for w in plan['weeks']]
plt.figure(figsize=(8, 3.3))
plt.bar(range(1, len(vols)+1), vols, color=BAR)
plt.title(f'Generated plan — {level} runner, 10K in 12 weeks'); plt.xlabel('week'); plt.ylabel('km')
plt.tight_layout(); plt.show()

In [ ]:
# show a few weeks as a schedule
DAYS={1:'Mon',2:'Tue',3:'Wed',4:'Thu',5:'Fri',6:'Sat',7:'Sun'}
for wk in (plan['weeks'][0], plan['weeks'][4], plan['weeks'][-1]):
    print(f"\nWeek {wk['weekNumber']} — {wk['targetDistanceKm']} km")
    for s in wk['sessions']:
        print(f"  {DAYS[s['dayOfWeek']]}  {s['sessionType']:10s} {s['distanceKm']} km")

**That is the whole fitness plan AI in one run:** the Random Forest classifies the runner, and the rule builder produces a safe, level-appropriate schedule — volume anchored to the runner, 10% build, a cutback every fourth week, and a taper into race day. The fitness estimate (separate notebook) would fill in the per-session paces.

## What to take to your report

- **This is the machine-learning component of the fitness plan AI:** unsupervised clustering to define levels from data, plus a **Random Forest** classifier (same family as the risk model) to predict them — ~94% on held-out runners.
- **Feature importance** is a real finding: training volume and consistency decide level; running experience (tenure) barely separates these runners.
- **Honest limitations to state:**
  - No external ground-truth level exists, so the Random Forest reproduces the clusters (cluster-then-classify). It is not validated against a labelled truth.
  - Cluster separation is weak (step 3) — three levels is a practical, data-supported choice, not a proven structure.
  - Endomondo runners skew experienced, so beginners (and the tenure signal) are under-represented versus your real users.
- **Deployment:** run the Random Forest in a small Python service (as planned for the risk model), or use the saved per-level medians as the thresholds in the Node rule classifier so both agree. The rule version stays the live fallback — cheap and explainable.